🟢 Débutant | ⏱ 90 min | 🔑 chemins, `Path`, `with open`, UTF-8, `csv`, `json`, `try/except/else`

# Module 08 — Fichiers et erreurs : lire le vrai fichier

> 🎬 **Regarde d'abord la vidéo du module** (scénario `python-debutant-08`, à venir). Elle montre le même code
> dans un fichier `.py` ; ici, dans le notebook, c'est pareil, cellule par cellule.

**Ce que tu sais déjà** : découper une ligne de texte avec `split` et convertir avec `float` (module 02), parcourir
avec `for` (04), ranger les relevés dans une liste de dictionnaires (06) et écrire des fonctions (07).

**Ce que tu vas apprendre** : jusqu'ici, les données étaient **recopiées à la main** dans les cellules. Un vrai
data analyst ne recopie rien : il **lit des fichiers**. Tu vas lire les 180 relevés de `meteo.csv`, nettoyer un
fichier abîmé, écrire tes résultats dans des fichiers, et **survivre aux erreurs** (fichier absent, valeur `N/A`).

**Comment lire ce notebook** :
- Exécute les cellules **dans l'ordre**, avec `Shift + Entrée`.
- Devant une cellule 🔮, **prédis d'abord** ce qu'elle va afficher, puis exécute et compare.
- Les cellules ⚠️ plantent **exprès** : c'est pour apprendre à lire les messages d'erreur.

## 1. Fichiers, dossiers et chemins

Un **fichier**, c'est un document rangé sur le disque : `meteo.csv` est un fichier. Un **dossier** (ou
répertoire) range des fichiers et d'autres dossiers, comme une chemise cartonnée dans une armoire. Tu les as
manipulés dans Git Bash au module 00 (`ls`, `cd`, `mkdir`).

Voici comment est rangé le parcours :

```
00-Parcours-Progressif/
├── donnees/
│   ├── meteo.csv                ← 180 relevés propres
│   ├── meteo_brut.csv           ← les mêmes, abîmés
│   └── open_meteo_toulouse.json
└── 08-fichiers-erreurs/
    ├── 01-cours.ipynb           ← tu es ici
    └── sorties/                 ← tes fichiers écrits iront là
```

Un **chemin**, c'est l'**adresse** d'un fichier. Un notebook travaille **depuis son propre dossier** (ici
`08-fichiers-erreurs/`) : c'est son **dossier de travail**. Un **chemin relatif** part de ce dossier de travail :

- `sorties/resume.json` : « dans le sous-dossier `sorties`, le fichier `resume.json` » ;
- `../donnees/meteo.csv` : `..` veut dire « remonter d'un cran » (le dossier parent, `00-Parcours-Progressif/`),
  puis entrer dans `donnees`, puis prendre `meteo.csv`.

C'est exactement `cd ..` de Git Bash. Les `/` séparent les étapes du chemin (même sous Windows, Python les comprend).

## 2. Sortir un outil de la boîte : `import` (le strict minimum)

Python est livré avec une grande **boîte à outils** : la **bibliothèque standard**. Ses outils sont rangés dans
des tiroirs appelés **modules**. Pour s'en servir, on les sort du tiroir avec `import`. Deux formes, que tu vas
utiliser telles quelles :

- `import csv` : « sors le tiroir `csv` » ; on écrit ensuite `csv.reader`, `csv.DictReader`...
  (le point veut dire « l'outil `reader` du tiroir `csv` ») ;
- `from pathlib import Path` : « dans le tiroir `pathlib`, prends seulement l'outil `Path` » ; on écrit ensuite
  directement `Path(...)`.

On met les `import` **en haut**, une fois pour toutes. Le module 09 expliquera tout cela en détail (et comment
installer de nouveaux tiroirs) ; pour l'instant, retiens juste ces deux lignes-recettes.

In [ ]:
import csv
import json
from pathlib import Path

print("Outils prêts : csv, json et Path")

## 3. `Path` : construire un chemin et vérifier qu'il existe

`Path("...")` transforme un texte en **chemin** : une adresse « intelligente », qui sait faire des choses utiles.
Trois gestes à connaître :

| Geste | Exemple | Ce que ça fait |
|---|---|---|
| Créer un chemin | `Path("../donnees")` | l'adresse du dossier `donnees` |
| Ajouter une étape avec `/` | `DONNEES / "meteo.csv"` | l'adresse de `meteo.csv` **dans** `donnees` |
| Vérifier | `chemin.exists()` | `True` si le fichier (ou dossier) existe vraiment, sinon `False` |

Le `/` entre un `Path` et un texte ne divise rien : il **colle les étapes** du chemin, comme les `/` de Git Bash.

Par convention, on écrit en MAJUSCULES une variable qui ne changera pas (une **constante**), comme `DONNEES`.

In [ ]:
DONNEES = Path("../donnees")
chemin_meteo = DONNEES / "meteo.csv"

print(chemin_meteo)
print(chemin_meteo.exists())

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
chemin_faux = DONNEES / "meteo_2025.csv"
print(chemin_faux.exists())
print((DONNEES / "meteo_brut.csv").exists())

<details><summary>👀 Voir l'explication</summary>

```
False
True
```
Aucun fichier `meteo_2025.csv` n'existe dans `donnees` : `.exists()` renvoie `False`, **sans planter**. C'est
une bonne façon de vérifier avant d'ouvrir. `meteo_brut.csv`, lui, existe.

</details>

Pour **créer un dossier**, on utilise `.mkdir()` (comme `mkdir` dans Git Bash). Avec `exist_ok=True`, Python
ne se plaint pas si le dossier existe déjà. Cette ligne-recette crée le dossier `sorties/` où tu écriras tes
fichiers ; tu la retrouveras au début de chaque exercice qui écrit un fichier.

In [ ]:
SORTIES = Path("sorties")
SORTIES.mkdir(exist_ok=True)
print(SORTIES.exists())

## 4. Ouvrir un fichier : `with open(...)`

Pour lire un fichier, il faut l'**ouvrir** avec la fonction intégrée `open` :

```python
with open(chemin_meteo, encoding="utf-8") as fichier:
    texte = fichier.read()
```

Mot par mot :
- `open(chemin_meteo, encoding="utf-8")` ouvre le fichier en **lecture** ;
- `encoding="utf-8"` dit **comment les lettres sont codées** dans le fichier (voir plus bas) ;
- `as fichier` donne un nom (une étiquette) au fichier ouvert ;
- `with ... :` puis un bloc indenté : **pendant** le bloc, le fichier est ouvert ; **à la fin** du bloc, Python
  le **referme tout seul**, même si une erreur survient.

Analogie : `with`, c'est **emprunter un classeur aux archives** avec la garantie qu'il sera rangé à la fin, même
si tu pars en courant. Un fichier oublié ouvert peut bloquer d'autres programmes : `with` évite ce souci.

`fichier.read()` lit **tout** le fichier d'un coup et renvoie une seule chaîne de caractères.

In [ ]:
with open(chemin_meteo, encoding="utf-8") as fichier:
    texte = fichier.read()

print(len(texte), "caractères")
print(texte[:120])

### L'encodage UTF-8

Un ordinateur ne stocke que des nombres. Un **encodage** est la **table de correspondance** entre les nombres et
les lettres. **UTF-8** est la table universelle d'aujourd'hui : elle connaît `é`, `î`, `°`, et même les émojis.
Nos fichiers sont en UTF-8. Si tu lis avec la mauvaise table, les accents se transforment en charabia :

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
with open(chemin_meteo, encoding="latin-1") as fichier:
    mauvais = fichier.read()

with open(chemin_meteo, encoding="utf-8") as fichier:
    bon = fichier.read()

print(mauvais[126:150])
print(bon[126:150])

<details><summary>👀 Voir l'explication</summary>

Avec `latin-1` (une vieille table), `Nîmes` devient `NÃ®mes` : le `î` est codé sur deux nombres en UTF-8, et
latin-1 les lit comme deux lettres. Avec `utf-8`, tout est correct.

**Règle** : écris **toujours** `encoding="utf-8"`. Sous Windows, sans cette précision, Python prend parfois une
autre table et les accents sont abîmés.

</details>

## 5. Lire un fichier ligne par ligne

Un fichier texte, c'est une suite de **lignes**. Le fichier ouvert se parcourt avec `for`, **une ligne à la fois** :
« pour chaque ligne du fichier ». Chaque ligne se termine par un caractère invisible de **retour à la ligne**,
qui s'écrit `\n`. On l'enlève avec `.strip()` (module 02).

In [ ]:
nb_lignes = 0
with open(chemin_meteo, encoding="utf-8") as fichier:
    for ligne in fichier:
        nb_lignes = nb_lignes + 1

print(nb_lignes, "lignes")

181 lignes : 1 ligne d'**en-tête** (les noms des colonnes) + 180 relevés.

Autre façon : `fichier.readlines()` renvoie **la liste de toutes les lignes**. Pratique pour prendre l'en-tête
(`lignes[0]`) et les données (`lignes[1:]`).

In [ ]:
with open(chemin_meteo, encoding="utf-8") as fichier:
    lignes = fichier.readlines()

print(repr(lignes[0]))
print(repr(lignes[0].strip()))
print(lignes[1].strip().split(","))

`repr(...)` affiche une chaîne **telle que Python la voit**, avec ses guillemets et ses caractères invisibles :
on voit le `\n` à la fin de la première ligne, puis il disparaît après `.strip()`.

La dernière ligne, c'est le geste du module 02 et de l'exercice `convertir_ligne` du module 07 : `split(",")`.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
with open(chemin_meteo, encoding="utf-8") as fichier:
    lignes = fichier.readlines()

morceaux = lignes[2].strip().split(",")
print(morceaux[1], morceaux[3])
print(type(morceaux[3]))

<details><summary>👀 Voir l'explication</summary>

`Montpellier 29.2` puis `<class 'str'>`. La 3ᵉ ligne (indice 2) est le relevé de Montpellier du 1ᵉʳ septembre.
**Attention** : `"29.2"` est un **texte**, pas un nombre ! Tout ce qu'on lit dans un fichier texte est du texte.
Pour calculer, il faut convertir avec `float(...)`.

</details>

### ⚠️ Erreur fréquente : utiliser le fichier après la fin du `with`

In [ ]:
with open(chemin_meteo, encoding="utf-8") as fichier:
    entete = fichier.readline()

reste = fichier.read()

**Lecture** : dernière ligne `ValueError: I/O operation on closed file.` → « opération d'entrée/sortie (*I/O*,
*input/output* : lecture ou écriture) sur un fichier **fermé** ».

La flèche `---->` pointe sur `reste = fichier.read()`, qui est **en dehors** du bloc `with` (pas indentée). À la
fin du bloc, le classeur a été rendu aux archives.

**Correction** : tout ce qui lit le fichier doit être **dans** le bloc, indenté sous le `with`.
(`fichier.readline()`, au singulier, lit une seule ligne : la suivante.)

### ⚠️ Erreur fréquente : un chemin faux

In [ ]:
with open("../donnees/meteo.cvs", encoding="utf-8") as fichier:
    texte = fichier.read()

**Lecture** : dernière ligne
`FileNotFoundError: [Errno 2] No such file or directory: '../donnees/meteo.cvs'`

- `FileNotFoundError` : « fichier non trouvé » ;
- `No such file or directory` : « aucun fichier ou dossier de ce nom » ;
- à la fin, entre apostrophes, **le chemin exact que Python a essayé** : relis-le lettre par lettre. Ici,
  `cvs` au lieu de `csv`.

Les trois causes habituelles : une **faute de frappe** ; un **mauvais point de départ** (le notebook n'est pas
dans le dossier que tu crois : vérifie avec `Path(".").resolve()`, qui affiche le chemin complet du dossier de
travail) ; un **`..` en trop ou en moins**. Vérifie avec `.exists()` étape par étape.

In [ ]:
print(Path(".").resolve())
print(Path("../donnees").exists(), Path("../donnees/meteo.csv").exists())

## 6. Le module `csv` : lire un tableau proprement

**CSV** (*Comma-Separated Values*, « valeurs séparées par des virgules ») est le format de tableau le plus simple :
une ligne = une ligne du tableau Excel, les colonnes séparées par des virgules. Excel sait l'ouvrir et l'enregistrer.

Découper avec `split(",")` marche... jusqu'au jour où une valeur contient elle-même une virgule (`"Toulouse, centre"`).
Le module `csv` gère tous ces cas pour toi. Deux lecteurs :

| Lecteur | Chaque ligne devient... | Accès |
|---|---|---|
| `csv.reader(fichier)` | une **liste** de textes | `ligne[1]` |
| `csv.DictReader(fichier)` | un **dictionnaire** dont les clés viennent de l'en-tête | `ligne["ville"]` |

Avec le module `csv`, on ouvre le fichier avec `newline=""` en plus : c'est la recette officielle, elle laisse
le module `csv` gérer lui-même les fins de ligne (sinon, sous Windows, des lignes vides peuvent apparaître à l'écriture).

`list(...)` range tout ce que le lecteur produit dans une liste (comme `list(range(3))` donne `[0, 1, 2]`).

In [ ]:
with open(chemin_meteo, encoding="utf-8", newline="") as fichier:
    lignes = list(csv.reader(fichier))

print(len(lignes))
print(lignes[0])
print(lignes[1])

Encore mieux pour nous : `csv.DictReader` fabrique **directement la liste de dictionnaires** du module 06, avec
les noms de colonnes comme clés. L'en-tête est consommé : il ne fait pas partie des données.

In [ ]:
with open(chemin_meteo, encoding="utf-8", newline="") as fichier:
    releves = list(csv.DictReader(fichier))

print(len(releves), "relevés")
print(releves[0])

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
premier = releves[0]
print(premier["ville"], premier["temp_max"])
print(premier["temp_max"] + "1")
print(float(premier["temp_max"]) + 1)

<details><summary>👀 Voir l'explication</summary>

```
Toulouse 25.9
25.91
26.9
```
Les valeurs d'un `DictReader` sont **toutes du texte**. `"25.9" + "1"` colle deux textes : `"25.91"` ! Après
`float(...)`, c'est un vrai nombre : `25.9 + 1 = 26.9`.

</details>

### ⚠️ Erreur fréquente : comparer un texte lu dans le fichier à un nombre

In [ ]:
if releves[0]["temp_max"] > 25:
    print("chaud")

**Lecture** : `TypeError: '>' not supported between instances of 'str' and 'int'`
→ « `>` n'est pas possible entre un `str` (texte) et un `int` (entier) ». Python refuse de comparer `"25.9"` et `25`.

**Correction** : convertir d'abord, `float(releves[0]["temp_max"]) > 25`. Le plus simple : convertir **une bonne
fois pour toutes** juste après la lecture, avec une fonction (module 07).

In [ ]:
def convertir_releve(ligne):
    """Transforme une ligne de DictReader (que du texte) en relevé avec de vrais nombres."""
    return {
        "date": ligne["date"],
        "ville": ligne["ville"],
        "temp_min": float(ligne["temp_min"]),
        "temp_max": float(ligne["temp_max"]),
        "pluie_mm": float(ligne["pluie_mm"]),
        "vent_kmh": int(ligne["vent_kmh"]),
    }

def charger_meteo(chemin):
    """Lit un CSV MétéoLocale (séparateur virgule) et renvoie la liste des relevés convertis."""
    releves = []
    with open(chemin, encoding="utf-8", newline="") as fichier:
        for ligne in csv.DictReader(fichier):
            releves.append(convertir_releve(ligne))
    return releves

releves = charger_meteo(chemin_meteo)
chauds = [r for r in releves if r["temp_max"] > 25]
print(len(releves), "relevés,", len(chauds), "journées au-dessus de 25 °C")

Regarde ce qu'on vient de faire : **toute** la donnée de septembre (180 relevés) chargée en trois lignes, puis
réutilisée avec les gestes des modules 05 à 07. Fini la saisie à la main.

### Un autre séparateur : `delimiter=";"`

En France, Excel enregistre souvent les CSV avec des **points-virgules** (parce que la virgule sert de virgule
décimale). C'est le cas de `meteo_brut.csv`. On le dit au lecteur avec l'argument nommé `delimiter=";"`
(*delimiter* = séparateur).

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
chemin_brut = DONNEES / "meteo_brut.csv"

with open(chemin_brut, encoding="utf-8", newline="") as fichier:
    sans_option = list(csv.reader(fichier))

with open(chemin_brut, encoding="utf-8", newline="") as fichier:
    avec_option = list(csv.reader(fichier, delimiter=";"))

print(len(sans_option[1]), sans_option[1])
print(len(avec_option[1]), avec_option[1])

<details><summary>👀 Voir l'explication</summary>

Sans l'option, le lecteur cherche des **virgules** : la ligne n'en contient pas, il la garde en **une seule
colonne** (`1`, une liste d'un seul long texte). Avec `delimiter=";"`, on obtient bien **6 colonnes**.
Quand un CSV ne donne qu'une colonne, pense au séparateur !

</details>

## 7. Écrire des fichiers

Pour **écrire**, on ouvre le fichier en mode **écriture** avec un 2ᵉ argument : `"w"` (*write*).

⚠️ `"w"` **crée** le fichier s'il n'existe pas, et l'**efface entièrement** s'il existe déjà. C'est pour ça qu'on
écrit dans `sorties/` et jamais dans `donnees/` : on ne risque pas d'écraser les originaux.

`fichier.write(texte)` écrit un texte. Il n'ajoute **pas** de retour à la ligne : c'est à toi de mettre `\n`.

In [ ]:
SORTIES.mkdir(exist_ok=True)
chemin_note = SORTIES / "note.txt"

with open(chemin_note, "w", encoding="utf-8") as fichier:
    fichier.write("Bulletin MétéoLocale\n")
    fichier.write("180 relevés lus\n")

with open(chemin_note, encoding="utf-8") as fichier:
    print(fichier.read())

### Écrire un CSV avec `csv.DictWriter`

`csv.DictWriter` fait l'inverse de `DictReader` : il écrit une liste de dictionnaires dans un CSV.
- `fieldnames=[...]` : la liste des colonnes, **dans l'ordre** voulu (*field names* = noms des champs) ;
- `.writeheader()` écrit la ligne d'en-tête ;
- `.writerow(dico)` écrit une ligne ; `.writerows(liste)` en écrit plusieurs d'un coup.

In [ ]:
toulouse = [r for r in releves if r["ville"] == "Toulouse"]
chemin_toulouse = SORTIES / "toulouse.csv"
colonnes = ["date", "ville", "temp_min", "temp_max", "pluie_mm", "vent_kmh"]

with open(chemin_toulouse, "w", encoding="utf-8", newline="") as fichier:
    ecrivain = csv.DictWriter(fichier, fieldnames=colonnes)
    ecrivain.writeheader()
    ecrivain.writerows(toulouse)

print(chemin_toulouse.exists())
print(len(charger_meteo(chemin_toulouse)), "relevés relus")

On **relit** toujours ce qu'on vient d'écrire pour vérifier : 30 relevés, c'est bien tout Toulouse.
Tu peux ouvrir `sorties/toulouse.csv` dans VS Code ou dans Excel pour le voir.

## 8. JSON : sauvegarder des dictionnaires et des listes

**JSON** (*JavaScript Object Notation*, prononcé « djé-sonne ») est un format texte qui ressemble **beaucoup** aux
dictionnaires et listes de Python : accolades `{}`, crochets `[]`, clés entre guillemets. C'est le format
qu'utilisent les sites web et les API pour échanger des données (module 09).

| Python | JSON |
|---|---|
| `dict` `{"a": 1}` | `{"a": 1}` (presque pareil !) |
| `list` `[1, 2]` | tableau `[1, 2]` |
| `True` / `False` / `None` | `true` / `false` / `null` |

Deux fonctions du module `json` :
- `json.dump(donnees, fichier, ensure_ascii=False, indent=2)` **écrit** (*dump* = déverser) ;
  `ensure_ascii=False` garde les accents lisibles, `indent=2` met en forme avec 2 espaces de retrait ;
- `json.load(fichier)` **relit** et renvoie les dictionnaires et listes Python.

In [ ]:
resume = {"mois": "2026-09", "nb_releves": len(releves), "villes": ["Toulouse", "Nîmes"], "complet": True}
chemin_json = SORTIES / "resume.json"

with open(chemin_json, "w", encoding="utf-8") as fichier:
    json.dump(resume, fichier, ensure_ascii=False, indent=2)

with open(chemin_json, encoding="utf-8") as fichier:
    print(fichier.read())

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
with open(chemin_json, encoding="utf-8") as fichier:
    relu = json.load(fichier)

print(relu["villes"][1])
print(relu == resume)

<details><summary>👀 Voir l'explication</summary>

`Nîmes` puis `True` : `json.load` a reconstruit **exactement** le même dictionnaire. On peut sauvegarder un
résultat aujourd'hui et le recharger demain dans un autre notebook.

</details>

## 9. Les erreurs prévisibles : `try` / `except` / `else`

Certaines erreurs **ne sont pas des bugs** : ce sont des situations normales qu'on peut prévoir. Le fichier du
jour n'est pas encore arrivé (`FileNotFoundError`), une case contient `N/A` au lieu d'un nombre (`ValueError`).
Plutôt que de laisser le programme planter, on prévoit un **plan B**.

📋 **Modèle à lire, pas à exécuter** (ce n'est pas une cellule de code : les lignes « plan A / plan B » sont des
explications, Python ne les comprendrait pas). Les vraies cellules à exécuter viennent juste après.

```text
try:
    ← ici, le plan A : ce qui peut échouer
except NomDeLErreur:
    ← ici, le plan B : ce qu'on fait si CETTE erreur arrive
else:
    ← ici, la suite, seulement si le plan A a réussi (facultatif)
```

Analogie : « **Essaie** (*try*) d'appeler Inès. **Si** elle ne répond pas (*except*), laisse un message.
**Sinon** (*else*), transmets-lui le bulletin. »

In [ ]:
try:
    valeur = float("N/A")
except ValueError:
    valeur = None
    print("Valeur manquante : on la remplace par None")

print(valeur)

Sans `try`, `float("N/A")` aurait planté avec `ValueError: could not convert string to float: 'N/A'` (vu au
module 02). Ici, Python **attrape** l'erreur et exécute le bloc `except`. Le programme continue.

Pour une **valeur manquante**, on utilise `None` (vu au module 07 : la valeur « rien »). Pour tester si une
variable vaut `None`, on écrit `valeur is None` (« valeur **est** None ») ou `valeur is not None`.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
def convertir_nombre(texte):
    """Convertit un texte en float ; renvoie None si c'est impossible (case vide, N/A...)."""
    try:
        nombre = float(texte.replace(",", "."))
    except ValueError:
        return None
    else:
        return nombre

print(convertir_nombre("25.9"), convertir_nombre("25,9"), convertir_nombre(""), convertir_nombre("N/A"))
print(convertir_nombre("N/A") is None)

<details><summary>👀 Voir l'explication</summary>

`25.9 25.9 None None` puis `True`. La virgule française est d'abord remplacée par un point ; `float("")` et
`float("N/A")` lèvent un `ValueError`, attrapé : on renvoie `None`. Le bloc `else` ne s'exécute que si la
conversion a réussi.

</details>

Le même geste protège l'ouverture d'un fichier qui n'est peut-être pas là :

In [ ]:
chemin_octobre = DONNEES / "meteo_octobre.csv"

try:
    releves_octobre = charger_meteo(chemin_octobre)
except FileNotFoundError:
    releves_octobre = []
    message = f"Pas encore de fichier {chemin_octobre.name} : bulletin d'octobre reporté."
else:
    message = f"{len(releves_octobre)} relevés d'octobre chargés."

print(message)

(`chemin.name` donne juste le nom du fichier, sans les dossiers.)

### ⚠️ Erreur fréquente : attraper la mauvaise erreur

`except` n'attrape **que** le type d'erreur écrit. Si une autre erreur arrive, elle plante normalement.

In [ ]:
try:
    valeur = float("N/A")
except FileNotFoundError:
    valeur = None

**Lecture** : le message montre `ValueError: could not convert string to float: 'N/A'`. Le `except` attendait un
`FileNotFoundError` : il n'a pas attrapé le `ValueError`, qui est passé à travers.

**Correction** : écrire le **bon** type d'erreur, celui que tu lis sur la dernière ligne du message :
`except ValueError:`.

**Bonne pratique** : n'écris jamais un `except` « attrape-tout » sans nom d'erreur. Tu cacherais tes vrais bugs
(une faute de frappe dans un nom de variable deviendrait invisible). Attrape **seulement** l'erreur que tu as prévue.

## 10. Tout ensemble : nettoyer `meteo_brut.csv`

`meteo_brut.csv` a été saisi à la main par un stagiaire pressé. Regarde ses défauts :

In [ ]:
with open(chemin_brut, encoding="utf-8", newline="") as fichier:
    brut = list(csv.DictReader(fichier, delimiter=";"))

print(len(brut), "lignes")
for ligne in brut[3:13]:
    print(ligne)

On repère : la virgule décimale (`'23,9'`), une case vide dans `pluie_mm` (`''`), `'N/A'` dans `vent_kmh`, des
villes en minuscules avec des espaces (`'  montpellier '`), et des **doublons** (la même ligne deux fois).

La méthode : **une fonction par défaut à corriger**, puis une boucle qui les applique.

In [ ]:
def nettoyer_ville(texte):
    """'  montpellier ' -> 'Montpellier'."""
    texte = texte.strip()
    return texte[0].upper() + texte[1:].lower()

def nettoyer_releve(ligne):
    """Une ligne brute (que du texte) -> un relevé propre (None pour une valeur manquante)."""
    return {
        "date": ligne["date"],
        "ville": nettoyer_ville(ligne["ville"]),
        "temp_min": convertir_nombre(ligne["temp_min"]),
        "temp_max": convertir_nombre(ligne["temp_max"]),
        "pluie_mm": convertir_nombre(ligne["pluie_mm"]),
        "vent_kmh": convertir_nombre(ligne["vent_kmh"]),
    }

propres = []
deja_vus = set()
for ligne in brut:
    releve = nettoyer_releve(ligne)
    cle = (releve["date"], releve["ville"])
    if cle not in deja_vus:
        deja_vus.add(cle)
        propres.append(releve)

manquants = [r for r in propres if r["pluie_mm"] is None]
print(len(brut), "lignes brutes ->", len(propres), "relevés propres")
print(len(manquants), "relevés sans pluie connue")

Pour les doublons, on retient chaque couple `(date, ville)` déjà vu dans un **ensemble** (`set`, module 06) : un
relevé dont la clé est déjà dans l'ensemble est ignoré. 185 lignes → 180 relevés.

Un tuple comme `(date, ville)` peut aussi servir de **clé de dictionnaire** : pratique pour retrouver le relevé
d'une ville **à une date** précise (une « fiche » repérée par deux rubriques à la fois).

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
officiel = {}
officiel[("2026-09-01", "Toulouse")] = 25.9
officiel[("2026-09-01", "Albi")] = 26.8
officiel[("2026-09-02", "Toulouse")] = 23.2

print(len(officiel))
print(officiel[("2026-09-01", "Albi")])

<details><summary>👀 Voir l'explication</summary>

`3` puis `26.8`. Trois clés différentes (deux dates pour Toulouse ne se mélangent pas) ; on lit la valeur en
redonnant **le même tuple** entre les crochets. Un tuple marche comme clé parce qu'on ne peut pas le modifier
(une liste, elle, est refusée comme clé).

</details>

Dernière étape : on **écrit** le résultat. `DictWriter` écrit `None` comme une case vide.

In [ ]:
with open(SORTIES / "meteo_propre.csv", "w", encoding="utf-8", newline="") as fichier:
    ecrivain = csv.DictWriter(fichier, fieldnames=colonnes)
    ecrivain.writeheader()
    ecrivain.writerows(propres)

print((SORTIES / "meteo_propre.csv").exists())

## 📌 À retenir

| Geste | Exemple |
|---|---|
| Sortir un outil | `import csv`, `import json`, `from pathlib import Path` |
| Chemin relatif | `DONNEES = Path("../donnees")` ; `DONNEES / "meteo.csv"` |
| Vérifier | `chemin.exists()` → `True` / `False` |
| Créer le dossier de sortie | `Path("sorties").mkdir(exist_ok=True)` |
| Lire tout le texte | `with open(chemin, encoding="utf-8") as fichier:` puis `fichier.read()` |
| Lire ligne par ligne | `for ligne in fichier:` + `ligne.strip()` ; ou `fichier.readlines()` |
| Lire un CSV | `with open(chemin, encoding="utf-8", newline="") as fichier:` puis `list(csv.DictReader(fichier))` |
| Autre séparateur | `csv.DictReader(fichier, delimiter=";")` |
| Convertir | tout ce qui est lu est du **texte** : `float(...)`, `int(...)` |
| Écrire un texte | `open(chemin, "w", encoding="utf-8")` + `fichier.write("...\n")` (⚠️ `"w"` efface) |
| Écrire un CSV | `csv.DictWriter(fichier, fieldnames=colonnes)`, `.writeheader()`, `.writerows(liste)` |
| JSON | `json.dump(donnees, fichier, ensure_ascii=False, indent=2)` ; `json.load(fichier)` |
| Plan B | `try:` ... `except ValueError:` ... `else:` ... |
| Valeur manquante | `None` ; tester avec `valeur is None` |

| Message d'erreur | Ce qu'il veut dire |
|---|---|
| `FileNotFoundError: [Errno 2] No such file or directory: '...'` | relis le chemin affiché : faute de frappe, `..` en trop ou en moins |
| `ValueError: I/O operation on closed file.` | tu lis le fichier en dehors du bloc `with` |
| `TypeError: '>' not supported between instances of 'str' and 'int'` | la valeur lue est un texte : convertis avec `float` |
| `ValueError: could not convert string to float: 'N/A'` | valeur manquante ou virgule décimale : `try/except` ou `replace` |

## 🧠 Mini-quiz

1. Le notebook est dans `08-fichiers-erreurs/`. Quel chemin relatif mène à `donnees/meteo_brut.csv` ?
2. Après `releves = list(csv.DictReader(fichier))`, que vaut `type(releves[0]["vent_kmh"])` ?
3. Tu ouvres `sorties/resume.json` avec `"w"`. Que devient l'ancien contenu ?
4. Un CSV lu avec `csv.reader` ne donne qu'une seule colonne par ligne. Quelle est la cause probable ?
5. Que fait ce code si `texte` vaut `"N/A"` ?
   `try: x = float(texte)` / `except ValueError: x = 0.0` / `else: print("ok")`

<details><summary>Réponses</summary>

1. `../donnees/meteo_brut.csv` (ou `Path("../donnees") / "meteo_brut.csv"`).
2. `<class 'str'>` : tout ce qu'on lit dans un CSV est du texte.
3. Il est **effacé** : `"w"` repart d'un fichier vide.
4. Le séparateur : le fichier utilise sans doute des `;`. Il faut `delimiter=";"`.
5. `x` vaut `0.0`, et rien ne s'affiche : le `else` ne s'exécute que si le `try` a réussi.

</details>

## ➡️ Et maintenant : ouvre `02-exercices.ipynb`